## Robot Perception (Part 2) -- Class Activity 3

### Object Pose Estimation

We will cover the following activities:
1. Camera calibration
2. Object pose estimation using fiducial markers (ArUco markers)
3. Object pose estimation using ArUco board


**Acknowledgements:** A good amount of this work builds on the tutorials provided by the **Robotics, Vision and Control** textbook Jupyter notebook [here](https://github.com/petercorke/RVC3-python/blob/d31ee173eab7acf43d58ee92f844289e34d279e4/notebooks/chap13.ipynb).

#### Preamble: Import relevant packages

In [ ]:
import numpy as np
from scipy import linalg
import matplotlib.pyplot as plt
np.set_printoptions(
    linewidth=120, formatter={
        'float': lambda x: f"{0:8.4g}" if abs(x) < 1e-10 else f"{x:8.4g}"})
np.random.seed(0)
from machinevisiontoolbox.base import *
from machinevisiontoolbox import *
from spatialmath.base import *
from spatialmath import *

### Section 1: Camera Calibration

Camera calibration is the process for determining the parameters of your specific camera. These include both the camera intrinsic parameters and the extrinsic parameters.

See this awesome resource to learn more about the checkerboard based calibration process: https://learnopencv.com/camera-calibration-using-opencv/

<img src="img\camera-calibration-flowchart.png" width="400">

Image source: https://learnopencv.com/camera-calibration-using-opencv/

**Step 1:** 

Use the **camera_capture.py** script to capture at least 10 images of the calibration checkerboard in different orientations in the camera view. Note where the image files are stored. 

**NOTE:** We will keep the camera fixed and move the checkerboard around to different orientations.

See image below for example:

<img src="img\ex-checkerboard.png" width="500">

To find the right camera interface:

1. On **Linux**, run ``v4l2-ctl --list-devices`` to get a list of all connected camera media devices
2. On **Windows**, simplest way is to try 0, 1, 2, etc. until you find the right one

**Step 2:** 

Read the calibration images


In [ ]:
images = ImageCollection("./img/calibration_imgs/*.png")

# display one of the calibration images
id = 3
images[id].disp() 

**Step 3:**

Compute the calibration results: camera intrinsic parameters (K), distortion, the pose (rotation and translation vectors) of the camera.

Read more here: https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html

**Acknowledgements:** Implementation code used from the machinevision-toolbox https://petercorke.github.io/machinevision-toolbox-python/camera.html

In [ ]:
# calibrate the camera using the checkerboard images
# the checkerboard gridshape is (9, 6) and the squaresize is 24mm
gridshape = (9, 6)
squaresize = 24e-3

criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)
# create set of feature points, like (0,0,0), (1,0,0), (2,0,0) ....,(6,5,0)
# these all have Z=0 since they are relative to the calibration target frame
objp = np.zeros((gridshape[0] * gridshape[1], 3), np.float32)
objp[:, :2] = (
    np.mgrid[0 : gridshape[0], 0 : gridshape[1]].T.reshape(-1, 2) * squaresize
)
print(f"3D object points of the known checkerboard corners:\n{objp}")


In [ ]:
# lists to store object points and image points from all the images
objpoints = []  # 3d point in real world space
imgpoints = []  # 2d points in image plane.
corner_images = []
valid = []

# loop through the images and find the corners
for i, image in enumerate(images):
    gray = image.mono().A

    # Find the chess board corners
    ret, corners = cv.findChessboardCorners(gray, gridshape, None)
    
    # If found, add object points, image points (after refining them)
    if ret:
        objpoints.append(objp)
        corners2 = cv.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners)

        # Draw the corners
        image = Image(image, copy=True)
        if not image.iscolor:
            image = image.colorize()
        corner_images.append(
            cv.drawChessboardCorners(image.A, gridshape, corners2, ret)
        )
        valid.append(i)

# calibrate the camera using the object points and image points
ret, K, distortion, rvecs, tvecs = cv.calibrateCamera(
    objpoints, imgpoints, gray.shape[::-1], None, None
)

print(f'The intrinsic matrix, K: \n{K} \n')
print(f'The distortion parameters: \n{distortion[0]} \n')  

In [ ]:
# show the corners recovered from one ot the calibration images
plt.imshow(corner_images[id])

**Step 4:**

Apply the intrinsic parameters and the lens distortion parameters to undistort the image

In [ ]:
distorted_img = images[id].A

undistorted_img = cv.undistort(distorted_img, K, distortion, None, None)
 
# titles = ['Original Image', 'Undistorted Image']
# plt.figure(figsize=(16, 6))
# plt.subplot(1,2,1)
# plt.imshow(distorted_img), plt.title(titles[0])
# plt.subplot(1,2,2)
# plt.imshow(undistorted_img), plt.title(titles[1])

**Extra Step:**

To obtain the camera extrinsic parameters (i.e., the position and orientation of the camera frame in a world frame), we can use the known positions of the checkerboard corners and the intrinsic parameters and distortion as follows.

The **goal of this code** is to compute the pose (position and orientation) and draw a 3D reference frame at the "origin" corner of the checkerboard.

In [ ]:
# we define a helper function to draw the 3D coordinate axes on the image using the projected points
def draw(img, corners, imgpts):
    """
    Draw 3D coordinate axes on an image using projected points.

    Arguments
    ----------
    img : numpy.ndarray
        The input image (BGR format) on which the axes will be drawn.

    corners : numpy.ndarray
        Detected 2D image points (e.g., chessboard corners)

    imgpts : numpy.ndarray
        Projected 3D axis points onto the image plane

    Returns
    -------
    numpy.ndarray
        The image with the three axes 
    """
    corner = tuple(corners[0].ravel().astype("int32"))
    imgpts = imgpts.astype("int32")
    img = cv.line(img, corner, tuple(imgpts[0].ravel()), (255,0,0), 5)
    img = cv.line(img, corner, tuple(imgpts[1].ravel()), (0,255,0), 5)
    img = cv.line(img, corner, tuple(imgpts[2].ravel()), (0,0,255), 5)
    return img

In [ ]:
# set the termination criteria for the cornerSubPix algorithm
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# create set of feature points, like (0,0,0), (1,0,0), (2,0,0) ....,(6,5,0)
# these all have Z=0 since they are relative to the calibration target frame
objpoints = np.zeros((gridshape[0] * gridshape[1], 3), np.float32)
objpoints[:, :2] = (
    np.mgrid[0 : gridshape[0], 0 : gridshape[1]].T.reshape(-1, 2) * squaresize
)

# set the 3D points for the axes to be drawn (in the calibration target frame)
axis_points = np.float32([[2,0,0], [0,2,0], [0,0,-2]]).reshape(-1,3) * squaresize

print(f"The axis points are:\n{axis_points}")

We use the **solvePnP** function which implements the **Perspective-n-Point** algorithm to retrieve the rotation and translation between the checkerboard and the camera frame that matches the 3D world positions and their corresponding 2D locations on the image plane.

<img src="img\opencv-pnp.jpg" width="400">

Image source: https://docs.opencv.org/4.x/d5/d1f/calib3d_solvePnP.html

In [ ]:
# Get the undistorted and convert to grayscale
gray = cv.cvtColor(undistorted_img, cv.COLOR_BGR2GRAY)

# Find checkerboard corners in pixel coordinates
ret, corners = cv.findChessboardCorners(gray, gridshape, None) 

if ret == True:
    # Refine the corner positions in pixel coordinates
    corners2 = cv.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
    
    # Given the known relative positions between objpoints and the corners2 (in pixels), 
    # find the rotation and translation between the checkerboard and the camera frame 
    # using the Perspective-n-Point (PnP) algorithm
    ret, rvecs, tvecs = cv.solvePnP(objpoints, corners2, K, distortion)
    
    # project 3D points of the coordinate axes to image plane using the estimated pose (rvecs and tvecs)
    imgpts, jac = cv.projectPoints(axis_points, rvecs, tvecs, K, distortion)

    img = undistorted_img.copy()
    img = draw(img, corners2, imgpts)
    plt.imshow(img)

print(f"Reference frame origin of the checkerboard is at pixel coordinates: {corners[0].ravel()}")
print(f"Projected axis points:\n{imgpts}")
print(f"Rotation vector: \n{rvecs}")
print(f"Translation vector: \n{tvecs}")

We can now recover both the **camera intrinsic parameters, K**, and the **extrinsic parameters, T** for this camera-checkerboard setup.

<img src="img\camera_params.png" width="1200">

In [ ]:
# Convert the rotation vector to a rotation matrix using Rodrigues
R, _ = cv.Rodrigues(rvecs)

# Combine the 3x3 rotation matrix and the 3x1 translation vector to compute
# the 4x4 camera transformation matrix
# T = np.hstack((R, tvecs.reshape(3, 1)))
# T = np.vstack((T, [0, 0, 0, 1]))

# print(f'Camera intrinsic matrix, K: \n{K} \n')
# print(f"Camera extrinsic (transformation) matrix, T: \n{T}")

**Practice Exercise:**

To build some intuition about the transformations between world, to camera to image to pixel coordinates and back, work on this exercise using the parameters we have computed.

In [ ]:
# TODO: Imagine an object at point (0.0, 0.0, 0.0), i.e., the origin of the world frame
# in our case, the reference frame of the checkerboard.
x = 0.0 * squaresize
y = 0.0 * squaresize
z = 0.0 * squaresize
obj_world = np.array([x, y, z, 1.0]) 

# TODO: Project this object from the world frame to the camera frame using T
# obj_camera = ??
# print(f"Object coordinate in the Camera Frame: {obj_camera[:3]}")

In [ ]:
# TODO: Project the object from the 3D camera coordinates to the 2D image plane (in pixel coordinates) using K
# obj_image = ??

# TODO: Normalize by the z-coordinate to get homogeneous coordinates
# obj_image_normalized = ??

# print(f"2D pixel coordinates [u~, v~, w~]: {obj_image}")
# print(f"2D pixel coordinates normalized [u, v]: {obj_image_normalized}")

In [ ]:
# TODO: Verify the transformations by drawing the object in the checkerboard and verifying the pixel coordinates

# # Draw a circle at the 2D pixel coordinates
color = (200,10,20)
# obj_pixel = (int(obj_image_normalized[0]), int(obj_image_normalized[1]))
# cv.circle(img, obj_pixel, radius=6, color=color, thickness=-1)

# # # Draw the trace lines
# cv.line(img, obj_pixel, (obj_pixel[0], 480), color, 1) # u axis
# cv.line(img, obj_pixel, (0, obj_pixel[1]), color, 1) # v axis

# plt.imshow(img)

### 2. Object pose estimation using ArUco markers

**Step 1:** 

Use the **camera_capture.py** script to capture an image of the ArUco marker. Note where the image files are stored. See image below for example:

<img src="img\ex-aruco-cubes.png" width="500">

**Step 2:**

Import the frame and then detect and perform pose estimation on the aruco markers in the frame.

In [ ]:
# Import the image frame stored from step 1
img = cv.imread("img/ex-aruco-cubes.png", cv.IMREAD_COLOR_RGB)

plt.imshow(img)

In [ ]:
# TODO: Undistort the frame using the parameters derived during calibration

undistorted_img = cv.undistort(img, K, distortion, None, None)
 
# titles = ['Original Image', 'Undistorted Image']
# plt.figure(figsize=(16, 6))
# plt.subplot(1,2,1)
# plt.imshow(img), plt.title(titles[0])
# plt.subplot(1,2,2)
# plt.imshow(undistorted_img), plt.title(titles[1])

In [ ]:
# TODO: Detect and estimate pose of the markers in the frame
# N.B.: the dictionary of the specific AruCo marker used, we're using the AprilTag 36h11 marker in this case

gray = cv.cvtColor(undistorted_img, cv.COLOR_BGR2GRAY)
aruco_dict = cv.aruco.getPredefinedDictionary(cv.aruco.DICT_APRILTAG_36H11)
parameters = cv.aruco.DetectorParameters()

# Create the ArUco detector
detector = cv.aruco.ArucoDetector(aruco_dict, parameters)
# Detect the markers
corners, ids, rejected = detector.detectMarkers(gray)

# Print the detected markers
print("Detected markers:", ids)
if ids is not None:
    cv.aruco.drawDetectedMarkers(undistorted_img, corners, ids)
    plt.imshow(undistorted_img)
    plt.show()


In [ ]:
# Estimate Pose
marker_length = 0.05 # Meters (actual size of your marker)

if ids is not None:
    rvecs, tvecs, _ = cv.aruco.estimatePoseSingleMarkers(
        corners, marker_length, K, distortion
    )
    
    # Draw axis for each marker
    axis_length = 0.05 # Length of the axes to be drawn
    for i in range(len(ids)):
        cv.drawFrameAxes(undistorted_img, K, distortion, rvecs[i], tvecs[i], axis_length)

    plt.imshow(undistorted_img)
    plt.show()

    print(f"Rotation vector: \n{rvecs}")
    print(f"Translation vector: \n{tvecs}")

Find the transformation matrix for one of the identified markers

In [ ]:
# Convert the rotation vector to a rotation matrix using Rodrigues
R, _ = cv.Rodrigues(rvecs[0])

# Combine the 3x3 rotation matrix and the 3x1 translation vector to compute
# the 4x4 camera transformation matrix
# T = np.hstack((R, tvecs[0].reshape(3, 1)))
# T = np.vstack((T, [0, 0, 0, 1]))

# print(f"Transformation Matrix, T: \n{T}")

### 3. Object pose estimation using ArUco board

The reason we might want to do this is to avoid needing to attach markers on individual objects in our operating space and use their relative pose to the AruCo board to estimate their pose with respect to the camera frame.

Resources: https://docs.opencv.org/3.4/db/da9/tutorial_aruco_board_detection.html

**Step 1:**

Use the **camera_capture.py** script to capture an image of the ArUco board with NO objects on it. Note where the image files are stored. See image below for example:

<img src="img\ex-aruco-board.png" width="500">

**Step 2:**

Import the frame and then detect and perform pose estimation on the aruco board to determine the pose of the reference frame of the board.

In [ ]:
# Import the image frame stored from step 1
img = cv.imread("img/ex-aruco-board.png", cv.IMREAD_COLOR_RGB)

plt.imshow(img)

In [ ]:
# TODO: Undistort the frame using the parameters derived during calibration

undistorted_img = cv.undistort(img, K, distortion, None, None)
 
titles = ['Original Image', 'Undistorted Image']
plt.figure(figsize=(16, 6))
plt.subplot(1,2,1)
plt.imshow(img), plt.title(titles[0])
plt.subplot(1,2,2)
plt.imshow(undistorted_img), plt.title(titles[1])

In [ ]:
gray = cv.cvtColor(undistorted_img, cv.COLOR_BGR2GRAY)

# Define the dictionary and board parameters
aruco_dict = cv.aruco.getPredefinedDictionary(cv.aruco.DICT_6X6_1000)
board = cv.aruco.GridBoard((5, 7), 0.04, 0.01, aruco_dict)
parameters = cv.aruco.DetectorParameters()

# Create the ArUco detector
detector = cv.aruco.ArucoDetector(aruco_dict, parameters)
corners, ids, rejected = detector.detectMarkers(gray)

plt.imshow(gray, cmap='gray')


In [ ]:
# Refine and estimate board pose (requires camera calibration)
if ids is not None:
#     # Optional: Refine markers that might have been missed
    corners, ids, rejected, recovered = detector.refineDetectedMarkers(
        gray, board, corners, ids, rejected
    )
    
    # Estimate pose (rvec and tvec) relative to the board
    # Requires camera intrinsic matrix and distortion coefficients from calibration
    obj_points, img_points = board.matchImagePoints(corners, ids)
    retval, rvec, tvec = cv.solvePnP(obj_points, img_points, K, distortion)

    print(f"Rotation vector: \n{rvec}")
    print(f"Translation vector: \n{tvec}")

In [ ]:
if ids is not None:
    retval, rvec, tvec = cv.aruco.estimatePoseBoard(corners, ids, board, K, distortion, rvec, tvec)

    if retval:
        # Draw axis for each marker
        axis_length = 0.05 # Length of the axes to be drawn
        for i in range(len(ids)):
            cv.drawFrameAxes(undistorted_img, K, distortion, rvecs, tvecs, axis_length)

        plt.imshow(undistorted_img)
        plt.show()